# Lightweight & Embedded Vector Stores: InMemoryVectorStore & Ephemeral Search

## 1. Overview and Core Motivation
Not every RAG application requires provisioning a distributed cloud database cluster or spinning up Docker containers.
For rapid prototyping, interactive experimentation, unit testing, and ephemeral agent session memory, **`InMemoryVectorStore`** provides a frictionless, zero-configuration vector index.

- **Zero Dependencies**: Backed natively by standard Python dictionaries and NumPy arrays.
- **Exact Exhaustive Search (Brute-Force)**: Calculates real-time pairwise cosine similarity using vector dot products ($O(N \cdot d)$).
- **Ideal For**: Small document sets (< 10,000 chunks), CI/CD pipelines, agent working memory, and sandbox evaluation.
- **Integrated Tech Stack**: Open-source HuggingFace `sentence-transformers/all-MiniLM-L6-v2` dense embeddings and ultra-fast **Groq LLM** via LangChain Expression Language (LCEL).

---

## 2. Architecture & Vector Retrieval Flow

![Vector Stores Architecture](workflow_vectorstores.png)

## 3. Technology Stack
- **Embeddings Model**: HuggingFace `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors).
- **Vector Store**: `langchain_core.vectorstores.InMemoryVectorStore`.
- **Language Model**: Groq `openai/gpt-oss-120b` via `init_chat_model('groq:openai/gpt-oss-120b')`.

In [ ]:
# Step 1: Environment Configuration & API Keys
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv, find_dotenv

# Search and load .env from current or parent directories
load_dotenv('../.env')
load_dotenv(find_dotenv())

# Verify Groq API Key
groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    print(f"GROQ_API_KEY detected successfully: {groq_key[:6]}...{groq_key[-4:]}")
else:
    print("Warning: GROQ_API_KEY not found in environment. Please set it in your .env file.")

In [ ]:
# Step 2: Initialize HuggingFace Embeddings and Groq LLM
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain.chat_models import init_chat_model

# 1. Embeddings: sentence-transformers/all-MiniLM-L6-v2 (384-dimensional dense vectors)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("HuggingFace all-MiniLM-L6-v2 embeddings initialized successfully.")

# 2. LLM: Groq ultra-fast LPU inference (openai/gpt-oss-120b or llama-3.3-70b-versatile)
try:
    llm = init_chat_model('groq:openai/gpt-oss-120b')
    test_res = llm.invoke("Respond with 'Groq Online' in 2 words.")
    print(f"Groq LLM Status: {test_res.content}")
except Exception as e:
    print(f"Falling back to groq:llama-3.3-70b-versatile due to: {e}")
    llm = init_chat_model('groq:llama-3.3-70b-versatile')

In [ ]:
# Step 3: Initialize InMemoryVectorStore with HuggingFace Embeddings
from langchain_core.vectorstores import InMemoryVectorStore

# The InMemoryVectorStore stores embeddings entirely in RAM memory as a Python dictionary.
# It computes exact cosine similarity on demand with numpy.
vector_store = InMemoryVectorStore(embedding=embedding_model)
print(f"InMemoryVectorStore initialized: {vector_store}")

In [ ]:
# Step 4: Define sample documents across diverse categories and metadata sources
from langchain_core.documents import Document

documents = [
    Document(
        page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
        metadata={"source": "tweet", "category": "food", "author": "alice"},
    ),
    Document(
        page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees and light rain.",
        metadata={"source": "news", "category": "weather", "author": "weather_channel"},
    ),
    Document(
        page_content="Building an exciting new production RAG system with LangChain and Groq - come check it out!",
        metadata={"source": "tweet", "category": "tech", "author": "developer_dan"},
    ),
    Document(
        page_content="Robbers broke into the downtown central bank and stole $1 million in cash overnight.",
        metadata={"source": "news", "category": "crime", "author": "city_gazette"},
    ),
    Document(
        page_content="Wow! That new sci-fi film was an amazing movie. The visual effects were groundbreaking.",
        metadata={"source": "tweet", "category": "entertainment", "author": "movie_fan"},
    ),
    Document(
        page_content="Is the new flagship smartphone worth the premium price tag? In-depth benchmarks and battery test results.",
        metadata={"source": "website", "category": "tech", "author": "tech_reviews"},
    ),
    Document(
        page_content="The top 10 soccer players in the world right now ranked by career goals and trophies.",
        metadata={"source": "website", "category": "sports", "author": "sports_weekly"},
    ),
    Document(
        page_content="LangGraph is the premier orchestration framework for building cyclical, stateful agentic AI applications!",
        metadata={"source": "tweet", "category": "tech", "author": "ai_researcher"},
    ),
    Document(
        page_content="The stock market plunged 500 points today due to global inflation concerns and interest rate hikes.",
        metadata={"source": "news", "category": "finance", "author": "wall_street_wire"},
    ),
    Document(
        page_content="Autonomous AI agents and Retrieval-Augmented Generation are transforming enterprise data search.",
        metadata={"source": "website", "category": "tech", "author": "ai_weekly"},
    ),
]

print(f"Created {len(documents)} structured Document objects with metadata tags.")

In [ ]:
# Step 5: Ingest documents into the InMemoryVectorStore
# This embeds all 10 documents using HuggingFace and populates the in-memory store
doc_ids = vector_store.add_documents(documents=documents)

print(f"Successfully added {len(doc_ids)} documents to the in-memory vector store.")
print(f"Sample generated Document IDs: {doc_ids[:3]}")

In [ ]:
# Step 6: Test basic similarity search
query = "How is the weather forecast looking?"
results = vector_store.similarity_search(query, k=2)

print(f"Query: '{query}'")
print("=" * 60)
for i, res in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content:  {res.page_content}")
    print(f"Metadata: {res.metadata}\n")

### Understanding InMemoryVectorStore Search Mechanics
`InMemoryVectorStore` computes real-time **Cosine Similarity** via dot products of normalized vectors:
$$\text{Cosine Similarity} = \frac{u \cdot v}{\|u\| \|v\|}$$
- Score range: **[-1.0 to 1.0]**
- **1.0** indicates identical semantic direction.
- Higher score = **MORE SIMILAR** (contrasting with Euclidean L2 distance where lower is closer).

In [ ]:
# Step 7: Similarity Search with numerical similarity scores
query = "What happened with the stolen money from the bank?"
results_with_scores = vector_store.similarity_search_with_score(query, k=3)

print(f"Query: '{query}'\n")
for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"Match {i} (Cosine Similarity Score: {score:.4f}):")
    print(f"Content:  {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

In [ ]:
# Step 8: Similarity search with Metadata Filtering
# In-memory stores filter results based on exact match criteria in document metadata
query = "What are the latest technology and framework updates?"

# Search only within tweets
filtered_results = vector_store.similarity_search(
    query,
    k=2,
    filter=lambda doc: doc.metadata.get("source") == "tweet" and doc.metadata.get("category") == "tech"
)

print(f"Filtered Query (category=tech, source=tweet): '{query}'\n")
for i, doc in enumerate(filtered_results, 1):
    print(f"Match {i}:")
    print(f"Content:  {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")

In [ ]:
# Step 9: Convert VectorStore to a LangChain Retriever
# The retriever abstracts the search implementation and integrates cleanly into chains
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

retrieved_docs = retriever.invoke("What did someone eat for breakfast?")
print(f"Retriever output ({len(retrieved_docs)} docs):")
for doc in retrieved_docs:
    print(f"- {doc.page_content} (Source: {doc.metadata['source']})")

In [ ]:
# Step 10: Build a Complete End-to-End RAG Chain with Groq LLM & LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Helper to format retrieved documents into clean context string
def format_docs(docs):
    return "\n\n".join(f"[Source: {doc.metadata.get('source', 'unknown')} | Category: {doc.metadata.get('category', 'general')}]\n{doc.page_content}" for doc in docs)

# Define strict, grounded RAG prompt
rag_prompt = ChatPromptTemplate.from_template("""You are a concise enterprise AI research assistant.
Answer the question using ONLY the provided context snippets below.
If the context does not contain the answer, explicitly state that the information is not available.

Context:
{context}

Question:
{question}

Answer:""")

# Assemble the LCEL Chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("End-to-End InMemory RAG Chain assembled successfully!")

# Test query execution
test_question = "What happened to the stock market and why?"
print(f"\nQuery: {test_question}")
print("=" * 60)
answer = rag_chain.invoke(test_question)
print(f"Groq LLM Answer:\n{answer}")

In [ ]:
# Step 11: Multi-Query Demonstration Function
def run_in_memory_rag(question: str):
    print(f"Question: {question}")
    print("-" * 50)
    
    # 1. Fetch source documents
    sources = retriever.invoke(question)
    
    # 2. Generate grounded answer
    answer = rag_chain.invoke(question)
    
    print(f"Answer:\n{answer}\n")
    print("Retrieved Context Chunks:")
    for i, doc in enumerate(sources, 1):
        print(f"  [{i}] ({doc.metadata['source']}) {doc.page_content[:100]}...")
    print("=" * 70 + "\n")

# Run tests
queries = [
    "Tell me about the weather tomorrow.",
    "Which frameworks are recommended for building agentic AI applications?",
    "What is the status of the bank robbery?"
]

for q in queries:
    run_in_memory_rag(q)